In [13]:
from langgraph.constants import END, START
from langgraph.graph import StateGraph
from typing import TypedDict
from IPython.display import display
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    # temperature=0.7,
    extra_body={
        "thinking": {
            "type": "disabled",
        }
    }
)

class OverAllState(TypedDict):
    topic: str
    poem: str
    joke: str

def node_a(state: OverAllState) -> OverAllState:
    poem = model.invoke([f"写一首关于{state['poem']}的诗"]).content
    return {
        "poem": poem,
        "topic": state["topic"],
        "joke": state["joke"],
    }

def node_b(state: OverAllState) -> OverAllState:
    joke  = model.invoke([f"写一个关于{state['joke']}的笑话"]).content
    return {
        "joke": joke,
        "topic": state["topic"],
        "poem": state["poem"],
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)  # 显式命名节点
builder.add_node("node_b", node_b)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "node_b")
builder.add_edge("node_b", END)

graph = builder.compile()
res = graph.invoke({"topic": "猴子", "poem": "", "joke": ""})

print(f"{res}")
display(graph)
print("Done")

OpenAIConnectionError: Connection error.